# Multi-Model Visual Comparison

Compare our top models side-by-side: best overall comp score, best individual
sub-metric scores (topo, SDice, VOI), and the pretrained baseline.

**Key questions:**
1. How do predictions differ visually between models?
2. For models with high topo but low SDice — which voxels cause the SDice penalty?
3. How does prediction thickness change across models?
4. Do better-topo models fragment the surface or just thin it?

**SDice Deep Dive:** For models with a significant SDice hit, we compute surface
boundary distances and show exactly which predicted surfaces are >tau=2 voxels from
GT (the voxels causing the penalty).

In [ ]:
import os
os.environ.setdefault('KERAS_BACKEND', 'torch')

try:
    import tensorflow as tf
    tf.config.set_visible_devices([], 'GPU')
except Exception:
    pass

import matplotlib
try:
    get_ipython()
    # Ensure inline rendering in Jupyter — this is critical for images to show
    from IPython import display
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['figure.max_open_warning'] = 50
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import tifffile
import time
import gc
import torch
from pathlib import Path
from scipy.ndimage import (
    binary_closing, binary_erosion, binary_dilation,
    generate_binary_structure, binary_propagation,
    label as scipy_label, zoom as scipy_zoom,
    distance_transform_edt,
)
import warnings
warnings.filterwarnings('ignore')

DRY_RUN = os.environ.get('DRY_RUN', '0') == '1'
if DRY_RUN:
    print('*** DRY RUN MODE -- using 2 volumes, fewer models ***')

ROOT = Path('/workspace/vesuvius-kaggle-competition')
TRAIN_IMG = ROOT / 'data' / 'train_images'
TRAIN_LBL = ROOT / 'data' / 'train_labels'
PLOT_DIR = ROOT / 'plots' / 'multi_model_comparison'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    """Save figure to disk AND display inline."""
    path = PLOT_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    # display.display(fig) is implicit with %matplotlib inline
    plt.show()
    plt.close(fig)
    print(f'  Saved: {path}')

print('Setup complete. Plots ->', PLOT_DIR)

## 1. Model Registry

Define all models to compare. Each model has:
- Weights path (for generating probmaps if not cached)
- Probmap cache directory
- Eval scores (from CSV logs)
- Why it's interesting

In [ ]:
# --- Model definitions ---
# Each model: weights_path, probmap_dir, scores dict, description
MODELS = {
    'pretrained': {
        'weights': ROOT / 'pretrained_weights/transunet/transunet.seresnext50.160px.comboloss.weights.h5',
        'probmap_dir': ROOT / 'data/transunet_probmaps',
        'scores': {'comp': 0.5526, 'topo': 0.2354, 'sdice': 0.8255, 'voi': 0.5517},
        'desc': 'Baseline pretrained (best SDice & VOI)',
        'color': '#2196F3',  # blue
    },
    'swa_70pre_30distsq': {
        'weights': ROOT / 'checkpoints/swa/swa_70pre_30distsq_ep5.weights.h5',
        'probmap_dir': ROOT / 'data/swa_70_30_probmaps',
        'scores': {'comp': 0.5530, 'topo': 0.2462, 'sdice': 0.8289, 'voi': 0.5401},
        'desc': 'Best comp (70% pretrained + 30% dist_sq_ep5)',
        'color': '#4CAF50',  # green
    },
    'frozen_boundary_ep10': {
        'weights': ROOT / 'checkpoints/transunet_frozen_boundary/transunet_ep10.weights.h5',
        'probmap_dir': ROOT / 'data/frozen_boundary_ep10_probmaps',
        'scores': {'comp': 0.5408, 'topo': 0.2642, 'sdice': 0.7861, 'voi': 0.5327},
        'desc': 'Best topo (frozen enc + dist_sq + boundary loss)',
        'color': '#FF9800',  # orange
    },
    'dist_sq_ep5': {
        'weights': ROOT / 'checkpoints/transunet_dist_sq/transunet_ep5.weights.h5',
        'probmap_dir': ROOT / 'data/dist_sq_probmaps',
        'scores': {'comp': 0.5246, 'topo': 0.2341, 'sdice': 0.7737, 'voi': 0.5246},
        'desc': 'Full fine-tune dist_sq (shows degradation)',
        'color': '#F44336',  # red
    },
}

if DRY_RUN:
    # Keep only models with existing probmaps for quick testing
    keep = ['pretrained', 'dist_sq_ep5']
    MODELS = {k: v for k, v in MODELS.items() if k in keep}

# Filter to models whose weights actually exist
available_models = {}
for name, cfg in MODELS.items():
    has_probmaps = cfg['probmap_dir'].exists() and len(list(cfg['probmap_dir'].glob('*.npy'))) > 0
    has_weights = cfg['weights'].exists()
    if has_probmaps or has_weights:
        available_models[name] = cfg
        status = 'probmaps cached' if has_probmaps else 'will run inference'
        print(f'  {name:30s} -> {status}')
    else:
        print(f'  {name:30s} -> SKIPPED (no weights or probmaps)')

MODELS = available_models
MODEL_NAMES = list(MODELS.keys())
print(f'\n{len(MODELS)} models available for comparison.')

## 2. Score Summary Table

All model scores from 24-volume cross-scroll evaluation. Bold = best per column.

In [ ]:
# Build score table
rows = []
for name, cfg in MODELS.items():
    s = cfg['scores']
    rows.append({
        'Model': name,
        'Comp': s['comp'],
        'Topo (30%)': s['topo'],
        'SDice (35%)': s['sdice'],
        'VOI (35%)': s['voi'],
        'Description': cfg['desc'],
    })

score_df = pd.DataFrame(rows)
print('Model Scores (24-vol cross-scroll eval, t_low=0.70, t_high=0.90):')
print()
print(score_df.to_string(index=False))

# Highlight best per metric
print()
for col in ['Comp', 'Topo (30%)', 'SDice (35%)', 'VOI (35%)']:
    best_idx = score_df[col].idxmax()
    print(f'  Best {col}: {score_df.loc[best_idx, "Model"]} ({score_df.loc[best_idx, col]:.4f})')

# SDice drop analysis
pretrained_sdice = MODELS['pretrained']['scores']['sdice'] if 'pretrained' in MODELS else 0.8255
print()
print('SDice vs pretrained:')
for name, cfg in MODELS.items():
    delta = cfg['scores']['sdice'] - pretrained_sdice
    flag = ' ** SIGNIFICANT DROP **' if delta < -0.03 else ''
    print(f'  {name:30s}: {delta:+.4f}{flag}')

## 3. Load / Generate Probmaps

For each model, load cached probmaps or run inference and cache them.

In [ ]:
def sigmoid_stable(x):
    x = np.asarray(x, dtype=np.float32)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out

def logsumexp2(a, b):
    m = np.maximum(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m) + 1e-12)

def build_anisotropic_struct(z_radius=3, xy_radius=2):
    sz = 2 * z_radius + 1
    sxy = 2 * xy_radius + 1
    struct = np.zeros((sz, sxy, sxy), dtype=bool)
    cy, cx = xy_radius, xy_radius
    for y in range(sxy):
        for x in range(sxy):
            if (y - cy) ** 2 + (x - cx) ** 2 <= xy_radius ** 2:
                struct[:, y, x] = True
    return struct

def postprocess(prob, t_low=0.70, t_high=0.90, z_radius=3, xy_radius=2, dust=100):
    strong = prob >= t_high
    weak = prob >= t_low
    struct = generate_binary_structure(3, 3)
    if not strong.any():
        return np.zeros_like(prob, dtype=np.uint8)
    mask = binary_propagation(strong, structure=struct, mask=weak)
    struct_close = build_anisotropic_struct(z_radius, xy_radius)
    mask = binary_closing(mask, structure=struct_close)
    labeled, n = scipy_label(mask)
    if n > 0:
        sizes = np.bincount(labeled.ravel())
        small = sizes < dust
        small[0] = False
        mask[small[labeled]] = 0
    return mask.astype(np.uint8)

def make_overlay(pred, lbl):
    """TP=green, FP=red, FN=blue. Ignores label=2."""
    mask = (lbl != 2)
    gt = (lbl == 1)
    pred_b = pred.astype(bool)
    rgb = np.zeros((*pred.shape, 3), dtype=np.float32)
    rgb[pred_b & gt & mask] = [0, 1, 0]      # TP
    rgb[pred_b & ~gt & mask] = [1, 0, 0]      # FP
    rgb[~pred_b & gt & mask] = [0, 0.4, 1]    # FN
    return rgb

def compute_surface(binary_mask):
    """Extract surface voxels (1-voxel boundary shell of the mask)."""
    if not binary_mask.any():
        return np.zeros_like(binary_mask)
    eroded = binary_erosion(binary_mask)
    return binary_mask & ~eroded

print('Helpers ready.')

In [ ]:
# Model loading (only done if needed for inference)
_loaded_model = None
_loaded_weights_path = None

def load_model_if_needed(weights_path):
    """Load TransUNet with given weights. Caches the model object."""
    global _loaded_model, _loaded_weights_path
    weights_path = str(weights_path)
    if _loaded_weights_path == weights_path:
        return _loaded_model
    
    from medicai.models import TransUNet
    from medicai.transforms import Compose, NormalizeIntensity
    from medicai.utils.inference import SlidingWindowInference
    
    if _loaded_model is None:
        model = TransUNet(
            input_shape=(160, 160, 160, 1),
            encoder_name='seresnext50',
            classifier_activation=None,
            num_classes=3,
        )
    else:
        model = _loaded_model['model']
    
    model.load_weights(weights_path)
    print(f'  Loaded weights: {Path(weights_path).name}')
    
    normalize = Compose([
        NormalizeIntensity(keys=['image'], nonzero=True, channel_wise=False),
    ])
    swi = SlidingWindowInference(
        model, num_classes=3, roi_size=(160,160,160),
        sw_batch_size=1, mode='gaussian', overlap=0.50,
    )
    
    _loaded_model = {'model': model, 'normalize': normalize, 'swi': swi}
    _loaded_weights_path = weights_path
    return _loaded_model

def predict_volume(weights_path, vol_path):
    """Run inference on a single volume, return (raw_img, prob)."""
    m = load_model_if_needed(weights_path)
    img = tifffile.imread(vol_path).astype(np.float32)
    vol_5d = img[None, ..., None]
    vol_5d = m['normalize']({'image': vol_5d})['image']
    with torch.no_grad():
        logits = np.asarray(m['swi'](vol_5d))[0]
    L0, L1, L2 = logits[..., 0], logits[..., 1], logits[..., 2]
    prob = sigmoid_stable(logsumexp2(L1, L2) - L0)
    del logits, vol_5d, L0, L1, L2
    gc.collect(); torch.cuda.empty_cache()
    return img, prob

print('Model loader ready.')

In [ ]:
# Select volumes from multiple scrolls
train_df = pd.read_csv(ROOT / 'data' / 'train.csv')
available_img = set(int(p.stem) for p in TRAIN_IMG.glob('*.tif'))
available_lbl = set(int(p.stem) for p in TRAIN_LBL.glob('*.tif'))
available = available_img & available_lbl
train_df = train_df[train_df.id.isin(available)]

if DRY_RUN:
    val_ids = train_df[train_df.scroll_id == 26002].id.tolist()
    VOLUME_PICKS = {'26002_a': val_ids[0], '26002_b': val_ids[1]}
else:
    VOLUME_PICKS = {}
    for scroll_id, n_pick in [(26002, 2), (35360, 1), (34117, 1)]:
        ids = train_df[train_df.scroll_id == scroll_id].id.tolist()
        for i in range(min(n_pick, len(ids))):
            VOLUME_PICKS[f'{scroll_id}_{chr(97+i)}'] = ids[i]

print(f'Selected {len(VOLUME_PICKS)} volumes:')
for key, vid in VOLUME_PICKS.items():
    print(f'  {key}: vol_id={vid}')

In [ ]:
# Load or generate probmaps for all models x volumes
# Structure: probmaps[model_name][vol_key] = prob_array
# Also: images[vol_key] = raw_img, labels[vol_key] = label

probmaps = {name: {} for name in MODEL_NAMES}
images = {}
labels = {}

for vol_key, vid in VOLUME_PICKS.items():
    print(f'\n=== Volume {vol_key} (id={vid}) ===')
    
    # Load raw image and label (once per volume)
    if vol_key not in images:
        images[vol_key] = tifffile.imread(TRAIN_IMG / f'{vid}.tif').astype(np.float32)
        labels[vol_key] = tifffile.imread(TRAIN_LBL / f'{vid}.tif')
    
    for model_name, cfg in MODELS.items():
        cache_path = cfg['probmap_dir'] / f'{vid}.npy'
        
        if cache_path.exists():
            prob = np.load(cache_path).astype(np.float32)
            print(f'  {model_name}: loaded cached probmap')
        elif cfg['weights'].exists():
            print(f'  {model_name}: running inference...')
            t0 = time.time()
            _, prob = predict_volume(cfg['weights'], TRAIN_IMG / f'{vid}.tif')
            elapsed = time.time() - t0
            print(f'    Done in {elapsed:.0f}s, prob range [{prob.min():.3f}, {prob.max():.3f}]')
            # Cache for reuse
            cfg['probmap_dir'].mkdir(parents=True, exist_ok=True)
            np.save(cache_path, prob.astype(np.float16))
            print(f'    Cached -> {cache_path}')
        else:
            print(f'  {model_name}: SKIPPED (no weights or cache)')
            continue
        
        probmaps[model_name][vol_key] = prob

print(f'\nLoaded probmaps for {len(MODEL_NAMES)} models x {len(VOLUME_PICKS)} volumes.')

## 4. Cross-Section Comparison

For each volume, one row per model: probmap | prediction | error overlay.
This is the key visualization — see how predictions differ between models.

In [ ]:
for vol_key, vid in VOLUME_PICKS.items():
    img = images[vol_key]
    lbl = labels[vol_key]
    scroll_id = vol_key.split('_')[0]
    
    # Find Z-slice with most foreground
    fg_counts = (lbl == 1).sum(axis=(1, 2))
    z = int(np.argmax(fg_counts))
    
    # Collect models that have probmaps for this volume
    vol_models = [(n, c) for n, c in MODELS.items() if vol_key in probmaps[n]]
    n_models = len(vol_models)
    if n_models == 0:
        continue
    
    fig, axes = plt.subplots(n_models + 1, 4, figsize=(24, 5 * (n_models + 1)))
    if n_models + 1 == 1:
        axes = axes[None, :]  # ensure 2D
    
    fig.suptitle(f'Model Comparison — Scroll {scroll_id}, Vol {vid}, Z={z}', fontsize=18, y=1.02)
    
    # Row 0: CT + GT
    axes[0, 0].imshow(img[z], cmap='gray')
    axes[0, 0].set_title('CT Image', fontsize=12)
    axes[0, 0].set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    gt_vis = np.zeros((*lbl[z].shape, 3), dtype=np.float32)
    gt_vis[lbl[z] == 1] = [0, 1, 0]
    gt_vis[lbl[z] == 2] = [0.5, 0.5, 0.5]
    axes[0, 1].imshow(gt_vis)
    axes[0, 1].set_title('GT (green=fg, gray=unlabeled)', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].axis('off')
    axes[0, 3].axis('off')
    
    # One row per model
    for i, (model_name, cfg) in enumerate(vol_models):
        row = i + 1
        prob = probmaps[model_name][vol_key]
        pred = postprocess(prob)
        overlay = make_overlay(pred[z], lbl[z])
        scores = cfg['scores']
        
        # Probmap
        im = axes[row, 0].imshow(prob[z], cmap='hot', vmin=0, vmax=1)
        axes[row, 0].set_title(f'Probmap (max={prob[z].max():.3f})', fontsize=11)
        axes[row, 0].set_ylabel(f'{model_name}\ncomp={scores["comp"]:.4f}',
                                fontsize=11, fontweight='bold')
        axes[row, 0].axis('off')
        
        # Binary prediction
        axes[row, 1].imshow(pred[z], cmap='gray')
        fg_pct = 100 * pred.sum() / pred.size
        axes[row, 1].set_title(f'Prediction (fg={fg_pct:.1f}%)', fontsize=11)
        axes[row, 1].axis('off')
        
        # Error overlay
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title('TP=green, FP=red, FN=blue', fontsize=11)
        axes[row, 2].axis('off')
        
        # Score bar
        metrics = ['topo', 'sdice', 'voi']
        vals = [scores[m] for m in metrics]
        colors_bar = ['#FF9800', '#2196F3', '#4CAF50']
        bars = axes[row, 3].barh(metrics, vals, color=colors_bar)
        axes[row, 3].set_xlim(0, 1)
        axes[row, 3].set_title('Sub-metric scores', fontsize=11)
        for bar, v in zip(bars, vals):
            axes[row, 3].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                              f'{v:.4f}', va='center', fontsize=10)
    
    plt.tight_layout()
    savefig(fig, f'comparison_{vol_key}')

## 5. Probability Distribution Comparison

Overlay probability histograms from all models on the same axes.
Shows how fine-tuning changes the probability landscape.

In [ ]:
n_vols = len(VOLUME_PICKS)
fig, axes = plt.subplots(2, n_vols, figsize=(7 * n_vols, 12))
if n_vols == 1:
    axes = axes[:, None]

for col, (vol_key, vid) in enumerate(VOLUME_PICKS.items()):
    lbl = labels[vol_key]
    fg_mask = (lbl == 1)
    bg_mask = (lbl == 0)
    scroll_id = vol_key.split('_')[0]
    
    for model_name, cfg in MODELS.items():
        if vol_key not in probmaps[model_name]:
            continue
        prob = probmaps[model_name][vol_key]
        color = cfg['color']
        
        # FG histogram (top row)
        fg_probs = prob[fg_mask]
        if fg_probs.size > 0:
            axes[0, col].hist(fg_probs.ravel(), bins=100, alpha=0.4, density=True,
                              range=(0, 1), color=color, label=model_name)
        
        # BG histogram (bottom row) — only show 0.3-1.0 range to see tail
        bg_probs = prob[bg_mask]
        if bg_probs.size > 0:
            bg_high = bg_probs[bg_probs > 0.1]
            if bg_high.size > 0:
                axes[1, col].hist(bg_high.ravel(), bins=100, alpha=0.4, density=True,
                                  range=(0.1, 1), color=color, label=model_name)
    
    # Threshold lines
    for ax_row in [0, 1]:
        for t, c, ls in [(0.70, 'orange', '--'), (0.90, 'red', '--')]:
            axes[ax_row, col].axvline(t, color=c, ls=ls, alpha=0.8)
    
    axes[0, col].set_title(f'Scroll {scroll_id} — FG probabilities', fontsize=12)
    axes[0, col].set_xlabel('Probability')
    axes[0, col].legend(fontsize=8)
    
    axes[1, col].set_title(f'Scroll {scroll_id} — BG false positives (p>0.1)', fontsize=12)
    axes[1, col].set_xlabel('Probability')
    axes[1, col].legend(fontsize=8)

fig.suptitle('Probability distributions across models', fontsize=16, y=1.02)
plt.tight_layout()
savefig(fig, 'probability_comparison')

## 6. Thickness Analysis

Measure how thick each model's predictions are. Thinner = better for SDice.
We measure thickness as the average number of consecutive FG voxels along the Z-axis
per column (x,y).

In [ ]:
def measure_thickness(binary_vol):
    """Measure mean thickness along Z-axis.
    For each (x,y) column, count consecutive FG runs and average their lengths.
    Returns: mean_thickness, fg_fraction, per-column max thickness map.
    """
    D, H, W = binary_vol.shape
    thickness_map = np.zeros((H, W), dtype=np.float32)
    all_runs = []
    
    for y in range(H):
        col = binary_vol[:, y, :]  # (D, W)
        for x in range(W):
            z_col = col[:, x]
            if not z_col.any():
                continue
            # Find runs of consecutive 1s
            changes = np.diff(z_col.astype(np.int8))
            starts = np.where(changes == 1)[0] + 1
            ends = np.where(changes == -1)[0] + 1
            # Handle edge cases
            if z_col[0] == 1:
                starts = np.concatenate([[0], starts])
            if z_col[-1] == 1:
                ends = np.concatenate([ends, [D]])
            runs = ends - starts
            if len(runs) > 0:
                thickness_map[y, x] = runs.max()
                all_runs.extend(runs.tolist())
    
    fg_frac = binary_vol.sum() / binary_vol.size
    mean_thick = np.mean(all_runs) if all_runs else 0
    median_thick = np.median(all_runs) if all_runs else 0
    return mean_thick, median_thick, fg_frac, thickness_map

# Compute thickness for each model on first volume
vol_key = list(VOLUME_PICKS.keys())[0]
vid = VOLUME_PICKS[vol_key]
lbl = labels[vol_key]

# GT thickness
gt_binary = (lbl == 1).astype(np.uint8)
gt_mean, gt_med, gt_fg, gt_thick_map = measure_thickness(gt_binary)
print(f'Ground truth: mean_thickness={gt_mean:.1f}, median={gt_med:.1f}, fg%={100*gt_fg:.2f}%')
print()

thickness_results = {}
thickness_maps = {}

for model_name, cfg in MODELS.items():
    if vol_key not in probmaps[model_name]:
        continue
    prob = probmaps[model_name][vol_key]
    pred = postprocess(prob)
    t0 = time.time()
    mean_t, med_t, fg_frac, t_map = measure_thickness(pred)
    elapsed = time.time() - t0
    thickness_results[model_name] = {'mean': mean_t, 'median': med_t, 'fg_frac': fg_frac}
    thickness_maps[model_name] = t_map
    ratio = mean_t / gt_mean if gt_mean > 0 else 0
    print(f'{model_name:30s}: mean={mean_t:.1f} ({ratio:.1f}x GT), '
          f'median={med_t:.1f}, fg%={100*fg_frac:.2f}% ({elapsed:.1f}s)')

# Thickness comparison plot
n_maps = len(thickness_maps) + 1  # +1 for GT
fig, axes = plt.subplots(1, n_maps, figsize=(6 * n_maps, 5))
if n_maps == 1:
    axes = [axes]

vmax = max(gt_thick_map.max(), max(m.max() for m in thickness_maps.values())) if thickness_maps else gt_thick_map.max()

im = axes[0].imshow(gt_thick_map, cmap='viridis', vmin=0, vmax=vmax)
axes[0].set_title(f'GT\nmean={gt_mean:.1f}', fontsize=12)
axes[0].axis('off')

for i, (model_name, t_map) in enumerate(thickness_maps.items()):
    r = thickness_results[model_name]
    im = axes[i+1].imshow(t_map, cmap='viridis', vmin=0, vmax=vmax)
    axes[i+1].set_title(f'{model_name}\nmean={r["mean"]:.1f} ({r["mean"]/gt_mean:.1f}x)', fontsize=11)
    axes[i+1].axis('off')

plt.colorbar(im, ax=axes[-1], shrink=0.8, label='Max Z-thickness (voxels)')
fig.suptitle(f'Prediction thickness maps — Vol {vid}', fontsize=16)
plt.tight_layout()
savefig(fig, 'thickness_comparison')

## 7. SDice Deep Dive

**For models with significant SDice drop vs pretrained** (i.e., good topo but bad SDice),
we identify exactly which voxels cause the penalty.

SDice (Surface Dice @ tau=2) measures surface boundary alignment:
- Extract surface voxels from both prediction and GT
- For each predicted surface voxel, find distance to nearest GT surface voxel
- Voxels within tau=2 are "matching"; voxels beyond tau=2 are "penalized"
- Also: GT surface voxels >tau from predicted surface are "missed"

We show cross-sections color-coded by surface distance to reveal where
and why SDice drops.

In [ ]:
TAU = 2.0  # Surface Dice tolerance

def sdice_surface_analysis(pred_binary, gt_binary, ignore_mask, tau=TAU):
    """Compute surface distance maps for SDice analysis.
    
    Returns:
        pred_surface: bool array of predicted surface voxels
        gt_surface: bool array of GT surface voxels
        pred_dist: distance from each pred surface voxel to nearest GT surface
        gt_dist: distance from each GT surface voxel to nearest pred surface
        pred_penalized: pred surface voxels with dist > tau (penalty contributors)
        gt_missed: GT surface voxels with dist > tau (missed by prediction)
    """
    valid = ~ignore_mask
    
    # Cast to bool to avoid uint8 bitwise NOT bug:
    # ~uint8(0)=255, ~uint8(1)=254 — both nonzero, breaks EDT
    pred_surface = compute_surface(pred_binary.astype(bool)) & valid
    gt_surface = compute_surface(gt_binary.astype(bool)) & valid
    
    # Distance from pred surface to nearest GT surface
    if gt_surface.any():
        dist_from_gt = distance_transform_edt(~gt_surface)
    else:
        dist_from_gt = np.full(pred_binary.shape, 999.0)
    pred_dist = dist_from_gt * pred_surface  # only at pred surface voxels
    pred_penalized = pred_surface & (dist_from_gt > tau)
    
    # Distance from GT surface to nearest pred surface
    if pred_surface.any():
        dist_from_pred = distance_transform_edt(~pred_surface)
    else:
        dist_from_pred = np.full(pred_binary.shape, 999.0)
    gt_dist = dist_from_pred * gt_surface
    gt_missed = gt_surface & (dist_from_pred > tau)
    
    # Summary stats
    n_pred_surf = pred_surface.sum()
    n_gt_surf = gt_surface.sum()
    n_pred_ok = (pred_surface & (dist_from_gt <= tau)).sum()
    n_gt_ok = (gt_surface & (dist_from_pred <= tau)).sum()
    
    sdice_approx = (n_pred_ok + n_gt_ok) / max(n_pred_surf + n_gt_surf, 1)
    
    return {
        'pred_surface': pred_surface,
        'gt_surface': gt_surface,
        'pred_dist': pred_dist,
        'gt_dist': gt_dist,
        'pred_penalized': pred_penalized,
        'gt_missed': gt_missed,
        'n_pred_surface': n_pred_surf,
        'n_gt_surface': n_gt_surf,
        'n_pred_penalized': pred_penalized.sum(),
        'n_gt_missed': gt_missed.sum(),
        'sdice_approx': sdice_approx,
    }

print('SDice analysis functions ready.')

In [ ]:
# Identify models with significant SDice drop
pretrained_sdice = MODELS['pretrained']['scores']['sdice'] if 'pretrained' in MODELS else 0.8255
SDICE_DROP_THRESHOLD = 0.03

sdice_drop_models = []
for name, cfg in MODELS.items():
    drop = pretrained_sdice - cfg['scores']['sdice']
    if drop > SDICE_DROP_THRESHOLD:
        sdice_drop_models.append(name)
        print(f'  {name}: SDice={cfg["scores"]["sdice"]:.4f} (drop={drop:.4f}, '
              f'topo={cfg["scores"]["topo"]:.4f})')

if not sdice_drop_models:
    print('No models with significant SDice drop (threshold={SDICE_DROP_THRESHOLD}).')
    print('Will analyze all non-pretrained models for comparison.')
    sdice_drop_models = [n for n in MODEL_NAMES if n != 'pretrained']

# Always include pretrained as reference
sdice_compare_models = ['pretrained'] + sdice_drop_models if 'pretrained' in MODELS else sdice_drop_models
sdice_compare_models = [m for m in sdice_compare_models if m in MODEL_NAMES]
print(f'\nWill compare SDice for: {sdice_compare_models}')

In [ ]:
# Run SDice surface analysis for each model x volume
sdice_results = {}  # [model_name][vol_key] = analysis dict

for model_name in sdice_compare_models:
    sdice_results[model_name] = {}
    for vol_key in VOLUME_PICKS:
        if vol_key not in probmaps[model_name]:
            continue
        
        prob = probmaps[model_name][vol_key]
        pred = postprocess(prob)
        lbl = labels[vol_key]
        gt = (lbl == 1).astype(np.uint8)
        ignore = (lbl == 2)
        
        t0 = time.time()
        result = sdice_surface_analysis(pred, gt, ignore)
        elapsed = time.time() - t0
        sdice_results[model_name][vol_key] = result
        
        print(f'{model_name:30s} {vol_key}: SDice~{result["sdice_approx"]:.4f}, '
              f'penalized={result["n_pred_penalized"]:,}, '
              f'missed={result["n_gt_missed"]:,} ({elapsed:.1f}s)')

print('\nSDice surface analysis complete.')

In [ ]:
# Visualize SDice penalty regions for each volume
# For each volume: one row per model showing the Z-slice with most penalized voxels

for vol_key in VOLUME_PICKS:
    vid = VOLUME_PICKS[vol_key]
    scroll_id = vol_key.split('_')[0]
    lbl = labels[vol_key]
    img = images[vol_key]
    
    # Find Z-slice with most SDice penalties across all models
    # Use the worst SDice-drop model to pick the slice
    worst_model = sdice_drop_models[0] if sdice_drop_models else sdice_compare_models[-1]
    if vol_key not in sdice_results.get(worst_model, {}):
        continue
    
    penalty_map = sdice_results[worst_model][vol_key]['pred_penalized']
    missed_map = sdice_results[worst_model][vol_key]['gt_missed']
    combined_penalty = penalty_map.astype(int) + missed_map.astype(int)
    penalty_per_z = combined_penalty.sum(axis=(1, 2))
    z_worst = int(np.argmax(penalty_per_z))
    
    # Also show the best-FG slice for comparison
    fg_counts = (lbl == 1).sum(axis=(1, 2))
    z_fg = int(np.argmax(fg_counts))
    
    z_slices = [z_fg, z_worst] if z_fg != z_worst else [z_fg]
    
    for z in z_slices:
        z_label = 'best_fg' if z == z_fg else 'worst_sdice'
        models_to_show = [m for m in sdice_compare_models if vol_key in sdice_results.get(m, {})]
        n_models = len(models_to_show)
        
        fig, axes = plt.subplots(n_models, 4, figsize=(28, 6 * n_models))
        if n_models == 1:
            axes = axes[None, :]
        
        fig.suptitle(
            f'SDice Surface Analysis — Scroll {scroll_id}, Vol {vid}, Z={z} ({z_label})',
            fontsize=16, y=1.02,
        )
        
        for i, model_name in enumerate(models_to_show):
            res = sdice_results[model_name][vol_key]
            cfg = MODELS[model_name]
            prob = probmaps[model_name][vol_key]
            pred = postprocess(prob)
            
            # Col 0: Prediction + GT outline
            pred_vis = np.zeros((*pred[z].shape, 3), dtype=np.float32)
            pred_vis[pred[z] == 1] = [0.6, 0.6, 0.6]  # prediction in gray
            gt_boundary_2d = compute_surface(np.expand_dims(lbl[z] == 1, 0))[0]
            pred_vis[gt_boundary_2d] = [0, 1, 0]  # GT boundary in green
            axes[i, 0].imshow(pred_vis)
            axes[i, 0].set_title('Pred (gray) + GT boundary (green)', fontsize=11)
            axes[i, 0].set_ylabel(
                f'{model_name}\nSDice={cfg["scores"]["sdice"]:.4f}',
                fontsize=11, fontweight='bold',
            )
            axes[i, 0].axis('off')
            
            # Col 1: Pred surface colored by distance to GT surface
            pred_surf_z = res['pred_surface'][z]
            pred_dist_z = res['pred_dist'][z]
            
            surf_rgb = np.zeros((*pred_surf_z.shape, 3), dtype=np.float32)
            if pred_surf_z.any():
                ok_mask = pred_surf_z & (pred_dist_z <= TAU)
                bad_mask = pred_surf_z & (pred_dist_z > TAU)
                surf_rgb[ok_mask] = [0, 0.8, 0]    # green = within tau
                surf_rgb[bad_mask] = [1, 0, 0]      # red = penalized
            # Show GT surface outline for reference
            gt_surf_z = res['gt_surface'][z]
            surf_rgb[gt_surf_z & ~pred_surf_z] = [0.3, 0.3, 1]  # blue = GT only
            
            axes[i, 1].imshow(surf_rgb)
            n_pen_z = bad_mask.sum() if pred_surf_z.any() else 0
            axes[i, 1].set_title(
                f'Pred surface: green=OK, red=penalized ({n_pen_z:,})\n'
                f'blue=GT surface (missed by pred)',
                fontsize=10,
            )
            axes[i, 1].axis('off')
            
            # Col 2: Distance heatmap on pred surface
            dist_display = np.full(pred_dist_z.shape, np.nan)
            dist_display[pred_surf_z] = pred_dist_z[pred_surf_z]
            im = axes[i, 2].imshow(dist_display, cmap='RdYlGn_r', vmin=0, vmax=8)
            axes[i, 2].set_title('Surface distance to GT (voxels)', fontsize=11)
            axes[i, 2].axis('off')
            plt.colorbar(im, ax=axes[i, 2], shrink=0.7)
            
            # Col 3: Missed GT surface (GT surf far from pred)
            gt_missed_z = res['gt_missed'][z]
            gt_ok_z = res['gt_surface'][z] & ~gt_missed_z
            missed_rgb = np.zeros((*gt_surf_z.shape, 3), dtype=np.float32)
            missed_rgb[gt_ok_z] = [0, 0.8, 0]      # green = GT surface near pred
            missed_rgb[gt_missed_z] = [1, 0.2, 0]   # red = GT surface missed
            # Show pred surface for reference
            pred_only = pred_surf_z & ~gt_surf_z
            missed_rgb[pred_only] = [0.3, 0.3, 0.3]  # gray = pred surface
            
            axes[i, 3].imshow(missed_rgb)
            n_miss_z = gt_missed_z.sum()
            axes[i, 3].set_title(
                f'GT surface: green=matched, red=missed ({n_miss_z:,})',
                fontsize=10,
            )
            axes[i, 3].axis('off')
        
        plt.tight_layout()
        savefig(fig, f'sdice_analysis_{vol_key}_z{z}_{z_label}')

## 8. SDice Penalty Breakdown

Quantify where the SDice penalty comes from: excess surface (FP boundary)
vs missed surface (FN boundary). This reveals whether the SDice drop is from
fragmentation (many small wrong surfaces) or from missing coverage.

In [ ]:
# Aggregate SDice stats across volumes
print(f'{"Model":30s} {"SDice":>8s} {"Pred Surf":>12s} {"GT Surf":>10s} '
      f'{"Penalized":>12s} {"Pen%":>6s} {"Missed":>10s} {"Miss%":>6s}')
print('-' * 100)

agg_stats = {}
for model_name in sdice_compare_models:
    total_pred_surf = 0
    total_gt_surf = 0
    total_penalized = 0
    total_missed = 0
    
    for vol_key, res in sdice_results.get(model_name, {}).items():
        total_pred_surf += res['n_pred_surface']
        total_gt_surf += res['n_gt_surface']
        total_penalized += res['n_pred_penalized']
        total_missed += res['n_gt_missed']
    
    pen_pct = 100 * total_penalized / max(total_pred_surf, 1)
    miss_pct = 100 * total_missed / max(total_gt_surf, 1)
    sdice_val = MODELS[model_name]['scores']['sdice']
    
    agg_stats[model_name] = {
        'pred_surf': total_pred_surf,
        'gt_surf': total_gt_surf,
        'penalized': total_penalized,
        'pen_pct': pen_pct,
        'missed': total_missed,
        'miss_pct': miss_pct,
    }
    
    print(f'{model_name:30s} {sdice_val:>8.4f} {total_pred_surf:>12,} {total_gt_surf:>10,} '
          f'{total_penalized:>12,} {pen_pct:>5.1f}% {total_missed:>10,} {miss_pct:>5.1f}%')

# Bar chart
if len(agg_stats) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    names = list(agg_stats.keys())
    pen_pcts = [agg_stats[n]['pen_pct'] for n in names]
    miss_pcts = [agg_stats[n]['miss_pct'] for n in names]
    colors = [MODELS[n]['color'] for n in names]
    
    ax1.barh(names, pen_pcts, color=colors, alpha=0.8)
    ax1.set_xlabel('% of pred surface penalized (dist > tau=2)')
    ax1.set_title('Excess surface (FP boundary)')
    for i, v in enumerate(pen_pcts):
        ax1.text(v + 0.3, i, f'{v:.1f}%', va='center')
    
    ax2.barh(names, miss_pcts, color=colors, alpha=0.8)
    ax2.set_xlabel('% of GT surface missed (dist > tau=2)')
    ax2.set_title('Missing surface (FN boundary)')
    for i, v in enumerate(miss_pcts):
        ax2.text(v + 0.3, i, f'{v:.1f}%', va='center')
    
    fig.suptitle('SDice penalty breakdown: excess vs missing surface', fontsize=14)
    plt.tight_layout()
    savefig(fig, 'sdice_penalty_breakdown')

## 9. Worst-SDice Volume Deep Dive

For models with SDice drop, find the volume where the penalty is largest
and show multiple Z-slices through the worst region.

In [ ]:
for model_name in sdice_drop_models:
    if model_name not in sdice_results or not sdice_results[model_name]:
        continue
    
    cfg = MODELS[model_name]
    
    # Find volume with most penalties
    worst_key = max(
        sdice_results[model_name].keys(),
        key=lambda k: sdice_results[model_name][k]['n_pred_penalized'] +
                      sdice_results[model_name][k]['n_gt_missed']
    )
    res = sdice_results[model_name][worst_key]
    vid = VOLUME_PICKS[worst_key]
    scroll_id = worst_key.split('_')[0]
    
    prob = probmaps[model_name][worst_key]
    pred = postprocess(prob)
    lbl = labels[worst_key]
    img = images[worst_key]
    
    # Find Z-range with most penalties
    penalty_per_z = (res['pred_penalized'].sum(axis=(1, 2)) +
                     res['gt_missed'].sum(axis=(1, 2)))
    z_peak = int(np.argmax(penalty_per_z))
    
    # Show 5 slices around the peak
    D = pred.shape[0]
    z_range = range(max(0, z_peak - 8), min(D, z_peak + 9), 4)
    z_slices = list(z_range)
    
    n_slices = len(z_slices)
    fig, axes = plt.subplots(n_slices, 5, figsize=(30, 5 * n_slices))
    if n_slices == 1:
        axes = axes[None, :]
    
    fig.suptitle(
        f'{model_name} — Worst SDice volume\n'
        f'Scroll {scroll_id}, Vol {vid} (penalized={res["n_pred_penalized"]:,}, '
        f'missed={res["n_gt_missed"]:,})',
        fontsize=14, y=1.02,
    )
    
    for i, z in enumerate(z_slices):
        # CT
        axes[i, 0].imshow(img[z], cmap='gray')
        axes[i, 0].set_ylabel(f'Z={z}', fontsize=11)
        axes[i, 0].axis('off')
        
        # Probmap
        axes[i, 1].imshow(prob[z], cmap='hot', vmin=0, vmax=1)
        axes[i, 1].axis('off')
        
        # Error overlay
        overlay = make_overlay(pred[z], lbl[z])
        axes[i, 2].imshow(overlay)
        axes[i, 2].axis('off')
        
        # SDice penalty: pred surface colored
        pred_surf_z = res['pred_surface'][z]
        pred_dist_z = res['pred_dist'][z]
        surf_rgb = np.zeros((*pred_surf_z.shape, 3), dtype=np.float32)
        if pred_surf_z.any():
            surf_rgb[pred_surf_z & (pred_dist_z <= TAU)] = [0, 0.8, 0]
            surf_rgb[pred_surf_z & (pred_dist_z > TAU)] = [1, 0, 0]
        gt_surf_z = res['gt_surface'][z]
        surf_rgb[gt_surf_z & ~pred_surf_z] = [0.3, 0.3, 1]
        axes[i, 3].imshow(surf_rgb)
        axes[i, 3].axis('off')
        
        # GT missed
        gt_missed_z = res['gt_missed'][z]
        gt_ok_z = res['gt_surface'][z] & ~gt_missed_z
        missed_rgb = np.zeros((*gt_surf_z.shape, 3), dtype=np.float32)
        missed_rgb[gt_ok_z] = [0, 0.8, 0]
        missed_rgb[gt_missed_z] = [1, 0.2, 0]
        axes[i, 4].imshow(missed_rgb)
        axes[i, 4].axis('off')
    
    # Column headers
    axes[0, 0].set_title('CT', fontsize=12)
    axes[0, 1].set_title('Probmap', fontsize=12)
    axes[0, 2].set_title('Error (G=TP R=FP B=FN)', fontsize=12)
    axes[0, 3].set_title('Pred surf (G=OK R=penalty)', fontsize=12)
    axes[0, 4].set_title('GT surf (G=matched R=missed)', fontsize=12)
    
    plt.tight_layout()
    savefig(fig, f'sdice_worst_{model_name}')

## 10. Multi-axis SDice view (Y and X slices)

SDice penalties may cluster along certain axes. Show coronal (Y) and sagittal (X)
slices for the SDice-drop models to understand the 3D structure of penalty regions.

In [ ]:
for model_name in sdice_drop_models[:1]:  # Just first SDice-drop model
    if model_name not in sdice_results:
        continue
    
    # Use first volume
    vol_key = list(sdice_results[model_name].keys())[0]
    vid = VOLUME_PICKS[vol_key]
    scroll_id = vol_key.split('_')[0]
    res = sdice_results[model_name][vol_key]
    prob = probmaps[model_name][vol_key]
    pred = postprocess(prob)
    img = images[vol_key]
    lbl = labels[vol_key]
    
    fig, axes = plt.subplots(3, 4, figsize=(28, 18))
    fig.suptitle(
        f'{model_name} — Multi-axis SDice view\n'
        f'Scroll {scroll_id}, Vol {vid}',
        fontsize=14,
    )
    
    for row, (axis, axis_name) in enumerate([
        (0, 'Z (axial)'), (1, 'Y (coronal)'), (2, 'X (sagittal)'),
    ]):
        # Find slice with most penalties along this axis
        sum_axes = tuple(i for i in range(3) if i != axis)
        penalty_per_slice = (
            res['pred_penalized'].sum(axis=sum_axes) +
            res['gt_missed'].sum(axis=sum_axes)
        )
        s = int(np.argmax(penalty_per_slice))
        
        slc = [slice(None)] * 3
        slc[axis] = s
        slc = tuple(slc)
        
        # CT
        axes[row, 0].imshow(img[slc], cmap='gray', aspect='auto')
        axes[row, 0].set_ylabel(f'{axis_name} s={s}', fontsize=11, fontweight='bold')
        axes[row, 0].axis('off')
        
        # Error overlay
        overlay = make_overlay(pred[slc], lbl[slc])
        axes[row, 1].imshow(overlay, aspect='auto')
        axes[row, 1].axis('off')
        
        # Pred surface penalty
        pred_surf = res['pred_surface'][slc]
        pred_dist = res['pred_dist'][slc]
        surf_rgb = np.zeros((*pred_surf.shape, 3), dtype=np.float32)
        if pred_surf.any():
            surf_rgb[pred_surf & (pred_dist <= TAU)] = [0, 0.8, 0]
            surf_rgb[pred_surf & (pred_dist > TAU)] = [1, 0, 0]
        gt_surf = res['gt_surface'][slc]
        surf_rgb[gt_surf & ~pred_surf] = [0.3, 0.3, 1]
        axes[row, 2].imshow(surf_rgb, aspect='auto')
        axes[row, 2].axis('off')
        
        # GT missed
        gt_missed = res['gt_missed'][slc]
        gt_ok = res['gt_surface'][slc] & ~gt_missed
        missed_rgb = np.zeros((*gt_surf.shape, 3), dtype=np.float32)
        missed_rgb[gt_ok] = [0, 0.8, 0]
        missed_rgb[gt_missed] = [1, 0.2, 0]
        axes[row, 3].imshow(missed_rgb, aspect='auto')
        axes[row, 3].axis('off')
    
    axes[0, 0].set_title('CT', fontsize=12)
    axes[0, 1].set_title('Error (G=TP R=FP B=FN)', fontsize=12)
    axes[0, 2].set_title('Pred surf (G=OK R=penalty)', fontsize=12)
    axes[0, 3].set_title('GT surf (G=matched R=missed)', fontsize=12)
    
    plt.tight_layout()
    savefig(fig, f'sdice_multiaxis_{model_name}')

## 11. Connected Component Analysis

Compare CC counts and size distributions across models.
Models with better topo should have CC counts closer to GT.

In [ ]:
print(f'{"Model":30s} {"Vol":>12s} {"GT_CC":>7s} {"Pred_CC":>8s} '
      f'{"Ratio":>7s} {"GT_fg%":>8s} {"Pred_fg%":>9s}')
print('-' * 90)

for vol_key in VOLUME_PICKS:
    lbl = labels[vol_key]
    gt = (lbl == 1).astype(np.uint8)
    _, n_gt = scipy_label(gt)
    gt_fg_pct = 100 * gt.sum() / gt.size
    vid = VOLUME_PICKS[vol_key]
    
    for model_name, cfg in MODELS.items():
        if vol_key not in probmaps[model_name]:
            continue
        prob = probmaps[model_name][vol_key]
        pred = postprocess(prob)
        _, n_pred = scipy_label(pred)
        pred_fg_pct = 100 * pred.sum() / pred.size
        ratio = n_pred / max(n_gt, 1)
        
        print(f'{model_name:30s} {vid:>12d} {n_gt:>7d} {n_pred:>8d} '
              f'{ratio:>6.2f}x {gt_fg_pct:>7.2f}% {pred_fg_pct:>8.2f}%')
    print()

In [ ]:
# CC size distributions: all models on same axes
vol_key = list(VOLUME_PICKS.keys())[0]
vid = VOLUME_PICKS[vol_key]
lbl = labels[vol_key]
gt = (lbl == 1).astype(np.uint8)
scroll_id = vol_key.split('_')[0]

fig, ax = plt.subplots(figsize=(12, 6))

# GT
gt_labeled, n_gt = scipy_label(gt)
gt_sizes = np.bincount(gt_labeled.ravel())[1:] if n_gt > 0 else []
if len(gt_sizes) > 0:
    ax.hist(np.log10(gt_sizes + 1), bins=40, alpha=0.5,
            label=f'GT ({n_gt} CCs)', color='black', histtype='step', linewidth=2)

for model_name, cfg in MODELS.items():
    if vol_key not in probmaps[model_name]:
        continue
    prob = probmaps[model_name][vol_key]
    pred = postprocess(prob)
    pred_labeled, n_pred = scipy_label(pred)
    pred_sizes = np.bincount(pred_labeled.ravel())[1:] if n_pred > 0 else []
    if len(pred_sizes) > 0:
        ax.hist(np.log10(pred_sizes + 1), bins=40, alpha=0.4,
                label=f'{model_name} ({n_pred} CCs)', color=cfg['color'])

ax.set_xlabel('log10(CC size)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'CC size distribution — Scroll {scroll_id}, Vol {vid}', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
savefig(fig, 'cc_size_comparison')

## 13. Summary

All models at a glance with key findings.

## 12. Post-Processing Sensitivity

**Key question:** Do different models need different PP parameters?
Thinner models may benefit from lower T_low (to preserve connectivity) while
thicker models need higher T_low (to thin). We sweep T_low and T_high for
each model and visualize how predictions change.

Also loads results from the full 82-volume PP sweeps if available.

In [ ]:
# --- Load existing full PP sweep results (82-volume, if available) ---
pp_sweep_files = {
    'pretrained': ROOT / 'logs' / 'postprocessing_sweep.csv',
    'dist_sq_ep5': ROOT / 'logs' / 'sweep_pp_dist_sq_results.csv',
}

print('=== Full 82-volume PP sweep results (from logs) ===')
print()
for model_name, csv_path in pp_sweep_files.items():
    if not csv_path.exists():
        print(f'{model_name}: no full sweep CSV found at {csv_path.name}')
        continue
    df = pd.read_csv(csv_path)
    if len(df) == 0:
        print(f'{model_name}: CSV is empty')
        continue
    # Sort by comp_score descending
    df = df.sort_values('comp_score', ascending=False)
    print(f'{model_name} — top 5 PP configs (of {len(df)}):')
    for _, row in df.head(5).iterrows():
        print(f'  {row["name"]:25s}: comp={row["comp_score"]:.4f}  '
              f'topo={row["topo"]:.4f}  sdice={row["sdice"]:.4f}  voi={row["voi"]:.4f}  '
              f'(t_low={row["t_low"]}, t_high={row["t_high"]}, z={row["z_radius"]}, xy={row["xy_radius"]})')
    print()

In [ ]:
# --- PP visual comparison: 4 configs per model ---
# Shows how predictions change with different PP params

PP_CONFIGS = [
    ('T=0.50/0.90\n(competitor)',  dict(t_low=0.50, t_high=0.90, z_radius=3, xy_radius=2, dust=100)),
    ('T=0.70/0.90\n(our default)', dict(t_low=0.70, t_high=0.90, z_radius=3, xy_radius=2, dust=100)),
    ('T=0.80/0.95\n(aggressive)',  dict(t_low=0.80, t_high=0.95, z_radius=3, xy_radius=2, dust=100)),
    ('T=0.50/0.80\n(permissive)',  dict(t_low=0.50, t_high=0.80, z_radius=1, xy_radius=1, dust=50)),
]

vol_key = list(VOLUME_PICKS.keys())[0]
vid = VOLUME_PICKS[vol_key]
lbl = labels[vol_key]
img = images[vol_key]
scroll_id = vol_key.split('_')[0]

fg_counts = (lbl == 1).sum(axis=(1, 2))
z = int(np.argmax(fg_counts))

vol_models = [(n, c) for n, c in MODELS.items() if vol_key in probmaps[n]]
n_models = len(vol_models)
n_pp = len(PP_CONFIGS)

fig, axes = plt.subplots(n_models, n_pp * 2, figsize=(6 * n_pp, 5 * n_models))
if n_models == 1:
    axes = axes[None, :]

fig.suptitle(
    f'PP Sensitivity — Scroll {scroll_id}, Vol {vid}, Z={z}\n'
    f'Top: prediction, Bottom: error overlay (per model)',
    fontsize=16, y=1.02,
)

for i, (model_name, cfg) in enumerate(vol_models):
    prob = probmaps[model_name][vol_key]
    
    for j, (pp_name, pp_params) in enumerate(PP_CONFIGS):
        pred = postprocess(prob, **pp_params)
        overlay = make_overlay(pred[z], lbl[z])
        fg_pct = 100 * pred.sum() / pred.size
        
        col_pred = j * 2
        col_over = j * 2 + 1
        
        # Prediction
        axes[i, col_pred].imshow(pred[z], cmap='gray')
        axes[i, col_pred].axis('off')
        if i == 0:
            axes[i, col_pred].set_title(f'{pp_name}\nfg%', fontsize=10)
        axes[i, col_pred].text(5, 15, f'{fg_pct:.1f}%', color='yellow',
                               fontsize=10, fontweight='bold',
                               bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        
        # Error overlay
        axes[i, col_over].imshow(overlay)
        axes[i, col_over].axis('off')
        if i == 0:
            axes[i, col_over].set_title('G=TP R=FP B=FN', fontsize=9)
    
    axes[i, 0].set_ylabel(f'{model_name}\ncomp={cfg["scores"]["comp"]:.4f}',
                           fontsize=11, fontweight='bold')

plt.tight_layout()
savefig(fig, 'pp_visual_comparison')

In [ ]:
# --- PP threshold sweep per model ---
# Sweep t_low from 0.30 to 0.85 with fixed t_high=0.90 and default closing.
# Measure fg%, CC count, and approximate SDice (from surface distance).
# This tells us if thinner models need different t_low than thicker ones.

T_LOW_VALUES = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85]
T_HIGH = 0.90

print(f'Sweeping t_low for each model on {len(VOLUME_PICKS)} volumes...')
print(f'Fixed: t_high={T_HIGH}, z_radius=3, xy_radius=2, dust=100')
print()

sweep_results = {}  # {model: {t_low: {fg_pct, n_cc, sdice_approx}}}

for model_name, cfg in MODELS.items():
    sweep_results[model_name] = {}
    for t_low in T_LOW_VALUES:
        total_fg = 0
        total_size = 0
        total_cc = 0
        total_pred_surf = 0
        total_gt_surf = 0
        total_pred_ok = 0
        total_gt_ok = 0
        n_vols = 0
        
        for vol_key in VOLUME_PICKS:
            if vol_key not in probmaps[model_name]:
                continue
            prob = probmaps[model_name][vol_key]
            pred = postprocess(prob, t_low=t_low, t_high=T_HIGH)
            lbl_v = labels[vol_key]
            gt_v = (lbl_v == 1).astype(np.uint8)
            ignore_v = (lbl_v == 2)
            valid_v = ~ignore_v
            
            total_fg += pred.sum()
            total_size += pred.size
            _, nc = scipy_label(pred)
            total_cc += nc
            
            # Approx SDice
            # Cast to bool to avoid uint8 bitwise NOT bug:
            # ~uint8(0)=255, ~uint8(1)=254 — both nonzero, breaks EDT
            pred_surf = compute_surface(pred.astype(bool)) & valid_v
            gt_surf = compute_surface(gt_v.astype(bool)) & valid_v
            
            if gt_surf.any() and pred_surf.any():
                d_from_gt = distance_transform_edt(~gt_surf)
                d_from_pred = distance_transform_edt(~pred_surf)
                total_pred_surf += pred_surf.sum()
                total_gt_surf += gt_surf.sum()
                total_pred_ok += (pred_surf & (d_from_gt <= TAU)).sum()
                total_gt_ok += (gt_surf & (d_from_pred <= TAU)).sum()
            
            n_vols += 1
        
        fg_pct = 100 * total_fg / max(total_size, 1)
        avg_cc = total_cc / max(n_vols, 1)
        sdice = (total_pred_ok + total_gt_ok) / max(total_pred_surf + total_gt_surf, 1)
        
        sweep_results[model_name][t_low] = {
            'fg_pct': fg_pct, 'avg_cc': avg_cc, 'sdice_approx': sdice,
        }
    
    # Print best t_low by approx SDice
    best_tlow = max(T_LOW_VALUES, key=lambda t: sweep_results[model_name][t]['sdice_approx'])
    best_sdice = sweep_results[model_name][best_tlow]['sdice_approx']
    print(f'{model_name:30s}: best t_low={best_tlow:.2f} (SDice~{best_sdice:.4f})')

# --- Plot: SDice vs t_low per model ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

for model_name, cfg in MODELS.items():
    color = cfg['color']
    tlows = sorted(sweep_results[model_name].keys())
    sdices = [sweep_results[model_name][t]['sdice_approx'] for t in tlows]
    fg_pcts = [sweep_results[model_name][t]['fg_pct'] for t in tlows]
    ccs = [sweep_results[model_name][t]['avg_cc'] for t in tlows]
    
    ax1.plot(tlows, sdices, 'o-', color=color, label=model_name, linewidth=2, markersize=6)
    ax2.plot(tlows, fg_pcts, 'o-', color=color, label=model_name, linewidth=2, markersize=6)
    ax3.plot(tlows, ccs, 'o-', color=color, label=model_name, linewidth=2, markersize=6)

ax1.set_xlabel('T_low', fontsize=12)
ax1.set_ylabel('Approx SDice', fontsize=12)
ax1.set_title('Surface Dice vs T_low', fontsize=13)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axvline(0.70, color='gray', ls=':', alpha=0.5, label='current default')

ax2.set_xlabel('T_low', fontsize=12)
ax2.set_ylabel('FG %', fontsize=12)
ax2.set_title('Foreground fraction vs T_low', fontsize=13)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

ax3.set_xlabel('T_low', fontsize=12)
ax3.set_ylabel('Avg CC count', fontsize=12)
ax3.set_title('Connected components vs T_low', fontsize=13)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

fig.suptitle('PP Threshold Sensitivity per Model', fontsize=16, y=1.02)
plt.tight_layout()
savefig(fig, 'pp_threshold_sweep')

In [ ]:
import glob

print('=' * 70)
print('MULTI-MODEL COMPARISON SUMMARY')
print('=' * 70)
print()

# Score table
print('Model Scores (24-vol cross-scroll eval):')
print(f'{"Model":30s} {"Comp":>8s} {"Topo":>8s} {"SDice":>8s} {"VOI":>8s}')
print('-' * 65)
for name, cfg in MODELS.items():
    s = cfg['scores']
    best_markers = ''
    if s['comp'] == max(c['scores']['comp'] for c in MODELS.values()):
        best_markers += ' [BEST COMP]'
    if s['topo'] == max(c['scores']['topo'] for c in MODELS.values()):
        best_markers += ' [BEST TOPO]'
    if s['sdice'] == max(c['scores']['sdice'] for c in MODELS.values()):
        best_markers += ' [BEST SDICE]'
    print(f'{name:30s} {s["comp"]:>8.4f} {s["topo"]:>8.4f} '
          f'{s["sdice"]:>8.4f} {s["voi"]:>8.4f}{best_markers}')

# Thickness summary
if thickness_results:
    print()
    print('Prediction thickness (first volume):')
    print(f'  GT: mean={gt_mean:.1f} voxels')
    for name, r in thickness_results.items():
        ratio = r['mean'] / gt_mean if gt_mean > 0 else 0
        print(f'  {name:30s}: mean={r["mean"]:.1f} ({ratio:.1f}x GT), fg%={100*r["fg_frac"]:.2f}%')

# SDice penalty summary
if agg_stats:
    print()
    print('SDice penalty sources:')
    for name, stats in agg_stats.items():
        print(f'  {name:30s}: {stats["pen_pct"]:.1f}% excess surface, '
              f'{stats["miss_pct"]:.1f}% missing surface')

print()
print(f'Plots saved to: {PLOT_DIR}')
plots = sorted(glob.glob(str(PLOT_DIR / '*.png')))
print(f'Generated {len(plots)} plots:')
for p in plots:
    print(f'  {Path(p).name}')

print()
print('Key findings:')
print('  - Check thickness maps: are fine-tuned models actually thinner?')
print('  - Check SDice penalty maps: excess surface or missing surface?')
print('  - Check CC distributions: fragmentation or consolidation?')
print('  - Compare probability distributions: sharper peaks in thinned models?')